In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
# os.environ["GOOGLE_CSE_ID"] = os.getenv("GOOGLE_CSE_ID")

### __Quickstart__

```Python
from langchain.agents import create_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_agent(
    model="gemini-3-flash-preview",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)
```

__While running the above code block you will get the following error__<br>
__ModuleNotFoundError: No module named 'langchain.agents'__ <br>
__ImportError: Could not import langchain-google-vertexai python package. Please install it with \`pip install langchain-google-vertexai\`__<br>
- This is because "from langchain.agents import create_agent" create_agent() is a high-level helper that tries to auto-detect the model provider from the model string. [“Gemini → Google → probably Vertex AI”].<br><br>

__Solution__<br>
__Correct way to use Gemini via Google GenAI (recommended)__<br>

❌ What NOT to do
- Do not pass model name as a string to create_agent
- Do not rely on auto-detection

✅ What TO do
- Explicitly create a ChatGoogleGenerativeAI model
- Pass the model object to the agent

- Step:1 Import Correct Model
```Python       
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_react_agent
```
- Step 2: Create the Gemini model explicitly. This forces LangChain to use Google GenAI, not Vertex AI.
```Python
model = ChatGoogleGenerativeAI(
    model_name="gemini-3-flash-preview",
    api_key="YOUR_API_KEY",
)
```

- Step 3: Create the agent
```Python
agent = create_react_agent(
    model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)
```

- Step 4: Run the agent
```Python
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)
```
---
__AGENT:__<br>
Agents combine language models with tools to create systems that can reason about tasks, decide which tools to use, and iteratively work towards solutions.<br>

__Core Components:__<br>
- Model [Static and Dynamic]
``` Python
from langchain.agents import create_agent
agent = create_agent("openai:gpt-5", tools=tools)
```
``` Python
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    model="gpt-5",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
    # ... (other params)
        )
agent = create_agent(model, tools=tools)
```





In [ ]:
# Example of instantiating an agent with a static model
# Type: 1
from langchain.agents import create_agent
tools = []
agent = create_agent("google_genai:gemini-3-flash-preview", tools=tools)

In [ ]:
# Exaample of instantiating static model and agent
# Type: 2
# Adding more control to agent
from langchain.agents import create_agent
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(
    model="gemini-3-flash-preview",
    temperature=0.1,
    max_tokens=1000,
    timeout=30
    # ... (other params)
)
agent = create_agent(model, tools=tools)

In [8]:
# Example of instantiating a dynamic model
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse


advanced_model = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
basic_model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

@wrap_model_call
def dynamic_model_selection(request: ModelRequest, handler) -> ModelResponse:
    """Choose model based on conversation complexity."""
    message_count = len(request.state["messages"])

    if message_count > 10:
        # Use an advanced model for longer conversations
        model = advanced_model
    else:
        model = basic_model

    return handler(request.override(model=model))

agent = create_agent(
    model=basic_model,  # Default model
    tools=tools,
    middleware=[dynamic_model_selection]
)

In [9]:
# Example of tool calling
from langchain.tools import tool
from langchain.agents import create_agent


@tool
def search(query: str) -> str:
    """Search for information."""
    return f"Results for: {query}"

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

agent = create_agent(model="google_genai:gemini-3-flash-preview", tools=[search, get_weather])

In [10]:
# Tool error handling

from langchain.agents import create_agent
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage


@wrap_tool_call
def handle_tool_errors(request, handler):
    """Handle tool execution errors with custom messages."""
    try:
        return handler(request)
    except Exception as e:
        # Return a custom error message to the model
        return ToolMessage(
            content=f"Tool error: Please check your input and try again. ({str(e)})",
            tool_call_id=request.tool_call["id"]
        )

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search, get_weather],
    middleware=[handle_tool_errors]
)

In [11]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage

literary_agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    system_prompt=SystemMessage(
        content=[
            {
                "type": "text",
                "text": "You are an AI assistant tasked with analyzing literary works.",
            },
            {
                "type": "text",
                "text": "<the entire contents of 'Pride and Prejudice'>",
                "cache_control": {"type": "ephemeral"}
            }
        ]
    )
)

"""
# This will make a call to the model
result = literary_agent.invoke(
    {"messages": [HumanMessage("Analyze the major themes in 'Pride and Prejudice'.")]}
)
"""

'\n# This will make a call to the model\nresult = literary_agent.invoke(\n    {"messages": [HumanMessage("Analyze the major themes in \'Pride and Prejudice\'.")]}\n)\n'

In [14]:
from typing import TypedDict

from langchain.agents import create_agent
from langchain.agents.middleware import dynamic_prompt, ModelRequest


class Context(TypedDict):
    user_role: str

@dynamic_prompt
def user_role_prompt(request: ModelRequest) -> str:
    """Generate system prompt based on user role."""
    user_role = request.runtime.context.get("user_role", "user")
    base_prompt = "You are a helpful assistant."

    if user_role == "expert":
        return f"{base_prompt} Provide detailed technical responses."
    elif user_role == "beginner":
        return f"{base_prompt} Explain concepts simply and avoid jargon."

    return base_prompt

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[get_weather],
    middleware=[user_role_prompt],
    context_schema=Context
)
"""
# The system prompt will be set dynamically based on context
result = agent.invoke(
    {"messages": [{"role": "user", "content": "Explain machine learning"}]},
    context={"user_role": "expert"}
)
"""

'\n# The system prompt will be set dynamically based on context\nresult = agent.invoke(\n    {"messages": [{"role": "user", "content": "Explain machine learning"}]},\n    context={"user_role": "expert"}\n)\n'

### __Models__

In [15]:
# Initialize chat model
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-3-flash-preview")

In [16]:
# More control using model configurations

model = init_chat_model(
    "google_genai:gemini-3-flash-preview",
    # Kwargs passed to the model:
    temperature=0.7,    # Lower temperature for more deterministic responses
    timeout=30,
    max_tokens=1000,
)

In [17]:
conversation = [
    {"role": "system", "content": "You are a helpful assistant that translates English to French."},
    {"role": "user", "content": "Translate: I love programming."},
    {"role": "assistant", "content": "J'adore la programmation."},
    {"role": "user", "content": "Translate: I love building applications."}
]

# response = model.invoke(conversation)
# print(response)  # AIMessage("J'adore créer des applications.")

In [18]:
from langchain.messages import HumanMessage, AIMessage, SystemMessage

conversation = [
    SystemMessage("You are a helpful assistant that translates English to French."),
    HumanMessage("Translate: I love programming."),
    AIMessage("J'adore la programmation."),
    HumanMessage("Translate: I love building applications.")
]

# response = model.invoke(conversation)
# print(response)  # AIMessage("J'adore créer des applications.")

In [19]:
# Batch processing
"""
responses = model.batch([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
])
for response in responses:
    print(response)

"""

'\nresponses = model.batch([\n    "Why do parrots have colorful feathers?",\n    "How do airplanes fly?",\n    "What is quantum computing?"\n])\nfor response in responses:\n    print(response)\n\n'

In [ ]:
"""
for response in model.batch_as_completed([
    "Why do parrots have colorful feathers?",
    "How do airplanes fly?",
    "What is quantum computing?"
]):
    print(response)
"""

In [21]:
"""
from langchain.tools import tool

@tool
def get_weather(location: str) -> str:
    \"""Get the weather at a location.\"""
    return f"It's sunny in {location}."


model_with_tools = model.bind_tools([get_weather])  

response = model_with_tools.invoke("What's the weather like in Boston?")
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")


"""

'\nfrom langchain.tools import tool\n\n@tool\ndef get_weather(location: str) -> str:\n    """Get the weather at a location."""\n    return f"It\'s sunny in {location}."\n\n\nmodel_with_tools = model.bind_tools([get_weather])  \n\nresponse = model_with_tools.invoke("What\'s the weather like in Boston?")\nfor tool_call in response.tool_calls:\n    # View tool calls made by the model\n    print(f"Tool: {tool_call[\'name\']}")\n    print(f"Args: {tool_call[\'args\']}")\n\n\n'